# Эксперимент 17: SLINet-style dB STFT + 2D CNN

**Статья:** SLINet: Dysphasia detection in children using deep neural network (SLINet: детекция дисфазии у детей с использованием глубоких нейронных сетей) 2021

**Ссылка:** [https://www.sciencedirect.com/science/article/abs/pii/S1746809421003955](https://www.sciencedirect.com/science/article/abs/pii/S1746809421003955)

**Краткое описание модели:** STFT magnitude в dB -> 2D CNN-классификатор в стиле SLINet.

**Содержание статьи:** SLINet-направление подчеркивает ценность спектральных карт для детекции детских речевых нарушений. dB-нормализация помогает стабилизировать динамический диапазон и улучшить обучение CNN. Эксперимент проверяет эту идею как переносимый baseline.

In [1]:
import sys
from pathlib import Path
import numpy as np
import time
from joblib import Parallel, delayed
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, precision_score, recall_score, classification_report

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))
sys.path.insert(0, str(exp_dir))

from shared import config, data_utils, train_utils
from shared.results_utils import save_result_csv
from model import get_model

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
n_letters = letters_train.shape[1]
N_FFT = 512
N_FREQ = N_FFT // 2 + 1
N_FRAMES = 320

def load_db(p):
    return data_utils.extract_stft_db(p, n_fft=N_FFT, max_frames=N_FRAMES)
X_train = np.stack(Parallel(n_jobs=-1)(delayed(load_db)(p) for p in paths_train))
X_val   = np.stack(Parallel(n_jobs=-1)(delayed(load_db)(p) for p in paths_val))
X_test  = np.stack(Parallel(n_jobs=-1)(delayed(load_db)(p) for p in paths_test))
m, s = X_train.mean(axis=(0,2,3), keepdims=True), X_train.std(axis=(0,2,3), keepdims=True)
s = np.where(s < 1e-6, 1.0, s)
X_train = (X_train - m) / s
X_val   = (X_val - m) / s
X_test  = (X_test - m) / s
print(f"dB STFT shape: (N, 1, {N_FREQ}, {N_FRAMES})")

dB STFT shape: (N, 1, 257, 320)


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
train_ds = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(letters_train).float(), torch.from_numpy(y_train).long())
val_ds   = TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(letters_val).float(), torch.from_numpy(y_val).long())
test_ds  = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(letters_test).float(), torch.from_numpy(y_test).long())
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

model = get_model(n_freq=N_FREQ, n_frames=N_FRAMES, num_classes=2, n_letters=n_letters, dropout=0.5).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n_params}")

Параметров: 552658


In [4]:
weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = train_utils.get_lr_scheduler(optimizer, patience=config.LR_SCHEDULER_PATIENCE, factor=config.LR_SCHEDULER_FACTOR)
early_stopping = train_utils.EarlyStopping(patience=config.EARLY_STOPPING_PATIENCE)
best_ckpt_path = exp_dir / "best_ckpt.pt"
best_f1 = -1.0

def eval_loader(loader):
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for x, letters, y in loader:
            x, letters, y = x.to(device), letters.to(device), y.to(device)
            pred = model(x, letters).argmax(dim=1)
            all_p.append(pred.cpu().numpy())
            all_t.append(y.cpu().numpy())
    return np.concatenate(all_p), np.concatenate(all_t)

N_EPOCHS = 50
t0 = time.perf_counter()
for epoch in range(N_EPOCHS):
    model.train()
    losses = []
    for x, letters, y in train_loader:
        x, letters, y = x.to(device), letters.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x, letters), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.DEFAULT_GRAD_CLIP)
        optimizer.step()
        losses.append(loss.item())
    vp, vt = eval_loader(val_loader)
    val_f1 = f1_score(vt, vp, average="macro")
    if val_f1 > best_f1:
        best_f1 = val_f1
        train_utils.save_best_checkpoint(model, best_ckpt_path)
    scheduler.step(val_f1)
    print(f"Epoch {epoch+1}/{N_EPOCHS}  loss={np.mean(losses):.4f}  val_f1_macro={val_f1:.4f}")
    if early_stopping.step(val_f1):
        print(f"Early stopping на эпохе {epoch+1}")
        break
train_time_sec = time.perf_counter() - t0
train_utils.load_best_checkpoint(model, best_ckpt_path, device)
print(f"Обучение: {train_time_sec:.1f} с. Лучший val_f1_macro={best_f1:.4f}")

Epoch 1/50  loss=5.6769  val_f1_macro=0.3644
Epoch 2/50  loss=2.6225  val_f1_macro=0.4149
Epoch 3/50  loss=1.9473  val_f1_macro=0.4192
Epoch 4/50  loss=2.2092  val_f1_macro=0.4398
Epoch 5/50  loss=1.8456  val_f1_macro=0.2705
Epoch 6/50  loss=1.7828  val_f1_macro=0.5150
Epoch 7/50  loss=1.6292  val_f1_macro=0.5051
Epoch 8/50  loss=1.6086  val_f1_macro=0.6662
Epoch 9/50  loss=1.7878  val_f1_macro=0.3292
Epoch 10/50  loss=1.7126  val_f1_macro=0.6297
Epoch 11/50  loss=1.0001  val_f1_macro=0.5598
Epoch 12/50  loss=0.8673  val_f1_macro=0.6428
Epoch 13/50  loss=0.8346  val_f1_macro=0.6511
Epoch 14/50  loss=0.6992  val_f1_macro=0.6854
Epoch 15/50  loss=0.7722  val_f1_macro=0.5539
Epoch 16/50  loss=0.6448  val_f1_macro=0.5875
Epoch 17/50  loss=0.5044  val_f1_macro=0.6345
Epoch 18/50  loss=0.4654  val_f1_macro=0.6826
Epoch 19/50  loss=0.4131  val_f1_macro=0.7130
Epoch 20/50  loss=0.3753  val_f1_macro=0.6701
Epoch 21/50  loss=0.3393  val_f1_macro=0.6695
Epoch 22/50  loss=0.2828  val_f1_macro=0.68

In [ ]:
model.eval()
all_logits = []
with torch.no_grad():
    for x, letters, _ in test_loader:
        x, letters = x.to(device), letters.to(device)
        all_logits.append(model(x, letters).cpu().numpy())
logits = np.concatenate(all_logits)
y_pred = np.argmax(logits, axis=1)
y_proba = torch.softmax(torch.from_numpy(logits), dim=1).numpy()[:, config.CLASS_BAD]
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, pos_label=config.CLASS_BAD, zero_division=0)
recall_bad = recall_score(y_test, y_pred, pos_label=config.CLASS_BAD, zero_division=0)
print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
save_result_csv(
    exp_dir=exp_dir, 
    experiment_id="exp_17_slinet_db", 
    experiment_name="dB STFT + CNN", 
    model="SLINet dB CNN", 
    accuracy=accuracy, 
    f1_macro=f1_macro, 
    f1_bad=f1_bad, 
    roc_auc=roc_auc, 
    precision_bad=precision_bad, 
    recall_bad=recall_bad, 
    notes="STFT magnitude dB", 
    num_params=n_params, 
    train_time_sec=train_time_sec
)

              precision    recall  f1-score   support

        good       0.77      0.76      0.77       282
         bad       0.52      0.53      0.53       135

    accuracy                           0.69       417
   macro avg       0.65      0.65      0.65       417
weighted avg       0.69      0.69      0.69       417

Результат сохранён в result.csv текущего эксперимента
